In [1]:
!pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate>=0.30.0" "trl>=0.9.0" pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 28.0 MB/s eta 0:00:00


In [1]:
!pip install -q wandb

In [2]:
import os

os.environ["WANDB_PROJECT"] = "nlp_final_math"    # 你的 project 名
os.environ["WANDB_PROJECT"] = "qwen_ft"    # 你的用户名 / team 名

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kimau197 (nlp_final_math) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import wandb
run = wandb.init(project="nlp_final")
print(wandb.run.project)   # 当前 run 的 project 名
print(wandb.run.entity)    # 当前 entity（你的用户名或 team）
print(wandb.run.name)      # 这次 run 的名字

nlp_final
Columbia_project
copper-water-1


## data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
from datasets import Dataset

# 读数据
df = pd.read_csv("/content/cot_data.csv")

# 只保留 teacher 解对的样本
df = df[df["correct"] == True].reset_index(drop=True)

print(df.head()[["question", "raw_output", "ground_truth"]])
print("总样本数:", len(df))

# 转成 HF Dataset
dataset = Dataset.from_pandas(df)

# 打乱一下再划分 80/20
dataset = dataset.shuffle(seed=42)
split = dataset.train_test_split(test_size=0.2)

train_dataset = split["train"]
eval_dataset  = split["test"]

print("train size:", len(train_dataset))
print("val size:", len(eval_dataset))

                                            question  \
0  Natalia sold clips to 48 of her friends in Apr...   
1  Weng earns $12 an hour for babysitting. Yester...   
2  Betty is saving money for a new wallet which c...   
3  Julie is reading a 120-page book. Yesterday, s...   
4  James writes a 3-page letter to 2 different fr...   

                                          raw_output ground_truth  
0  <think>\nLet me understand the problem: Natali...           72  
1  <think>\nLet me understand the problem: Weng e...           10  
2  <think>\nLet me understand the problem careful...            5  
3  <think>\nLet me understand the problem: Julie ...           42  
4  <think>\nLet me understand the problem: "James...          624  
总样本数: 7131
train size: 5704
val size: 1427


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Math-1.5B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
print("model loaded on:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

model loaded on: cuda:0


In [ ]:
SYSTEM_PROMPT = (
    "You are a mathematical reasoning assistant. "
    "Solve problems using step-by-step reasoning with clear explanations. "
    "Always provide your final answer in \\boxed{} format."
)

def formatting_prompts(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                example["question"]
                + "\nPlease reason step by step, and put your final answer within \\boxed{}."
            ),
        },
        {
            "role": "assistant",
            "content": example["raw_output"],
        },
    ]
    # 返回一个已经拼好的对话字符串（Qwen 模板）
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # 不加额外的 assistant 起始
    )
    return text

In [7]:
print(formatting_prompts(train_dataset[0]))

<|im_start|>system
You are a helpful math tutor. Solve math problems step by step. Use clear natural language reasoning and math notation. The final answer must be exact (no decimal approximation) and placed in \boxed{} on the last line.<|im_end|>
<|im_start|>user
Problem:
A retailer sells any shirt for the same price and any pair of pants for the same price, but the price of shirts and pants are not the same. Say you bought 2 shirts and 3 pairs of pants for $120 in total, then you realized you didn't need the extra pants. What's the price of 1 shirt at this place if when you returned all 3 pairs of pants you were refunded 25% of what you originally paid?

Please think step by step and then give the final answer in \boxed{} on the last line.<|im_end|>
<|im_start|>assistant
<think>
Let me understand the problem. We have 2 shirts at price \(s\) each and 3 pairs of pants at price \(p\) each, with \(2s + 3p = 120\).

The key is the refund: returned all 3 pairs of pants and got refunded 25%

In [ ]:
import gc, torch

del trainer
del model
gc.collect()
torch.cuda.empty_cache()

NameError: name 'trainer' is not defined

## SFT

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = "/content/drive/MyDrive/Columbia/nlp/final/model/qwen25_math_1_5b_sft"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,                 # 3 轮就够
    per_device_train_batch_size=8,     # 80G 完全扛得住
    gradient_accumulation_steps=3,      # 有效 batch = 16
    learning_rate=5e-6,                 # 比 1e-5 稍小，更稳
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True,                          # A100 用 bf16
    gradient_checkpointing=False,       # 1.5B + 80G 不需要省显存，关掉更快
    weight_decay=0.01,
    lr_scheduler_type="cosine",         # 比 linear 平滑一点
    optim="adamw_torch",
    report_to="none",
    dataloader_num_workers=4,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    formatting_func=formatting_prompts,
    args=training_args,

)

trainer.train()

# 保存最终模型
save_dir = f"{output_dir}/final_model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)

Applying formatting function to train dataset:   0%|          | 0/7131 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/7131 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7131 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7131 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,1.436500
40,1.345000
60,1.165800
80,1.017100
100,0.915100
120,0.832400
140,0.785900
160,0.769100
180,0.755400
200,0.731000


Saved to: /content/drive/MyDrive/Columbia/nlp/final/model/qwen25_math_1_5b_sft/final_model


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
import torch

model_path = "/content/drive/MyDrive/Columbia/nlp/final/model/qwen25_math_1_5b_sft/final_model"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

class StopOnBoxedAnswer(StoppingCriteria):
    """Halts generation once a \\boxed{} answer is produced."""

    def __init__(self, tokenizer, prompt_token_len: int):
        self.tokenizer = tokenizer
        self.prompt_token_len = prompt_token_len

    def _has_boxed_answer(self, text: str) -> bool:
        idx = text.rfind("\\boxed{")
        if idx == -1:
            return False
        return "}" in text[idx:]

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        generated_ids = input_ids[0][self.prompt_token_len :]
        if generated_ids.numel() == 0:
            return False
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        if self._has_boxed_answer(text):
            return True
        return False

def solve(problem):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem + "\nPlease reason step by step, and put your final answer within \\boxed{}."},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_length = inputs['input_ids'].shape[1]
    
    stopping_criteria = StoppingCriteriaList([StopOnBoxedAnswer(tokenizer, prompt_length)])
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            stopping_criteria=stopping_criteria,
        )
    print(tokenizer.decode(out[0], skip_special_tokens=True))

test_q = "Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?"
solve(test_q)

system
You are a helpful math tutor. Solve math problems step by step. Use clear natural language reasoning and math notation. The final answer must be exact (no decimal approximation) and placed in \boxed{} on the last line.
user
Problem:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Please think step by step and then give the final answer in \boxed{} on the last line.
assistant
<think>
Let me understand the problem: Josh buys a house for $80,000 and spends $50,000 on repairs. The repairs increase the house's value by 150%. We need to find the profit, which is the final value minus the total cost (initial house price + repairs).

First, calculate the increase in value: 150% of $80,000 = 1.5 × 80,000 = $120,000. So, the new value of the house is $80,000 + $120,000 = $200,000.

Total cost = $80,000 (initial house) + $50,000 (repairs) = $130,000.

Profi

## LoRA




In [ ]:
import gc, torch

del trainer
del model
gc.collect()
torch.cuda.empty_cache()

In [8]:
from peft import LoraConfig, TaskType

In [9]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # 因果语言模型
    r=16,                          # rank，8/16 都可以，16 稍微强一点
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none")

In [12]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = "/content/drive/MyDrive/Columbia/nlp/final/model/qwen25_math_1_5b_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=3,
    learning_rate=1e-4,
    warmup_ratio=0.03,

    logging_steps=10,              # 每 20 个 train step log 一次 train/loss
    logging_first_step=True,       # 第 1 步也 log 一次（可选）

    eval_strategy="steps",   # ✅ 改成按 step eval
    eval_steps=10,                 # ✅ 每 20 个 step 跑一次 eval，算 eval_loss
    save_strategy="steps",         # （可选）也按 step 存最优模型
    save_steps=10,                 # 和 eval_steps 对齐
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    bf16=True,
    gradient_checkpointing=False,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    report_to=["wandb"],
    run_name="qwen25_math_lora",
    dataloader_num_workers=4,

)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=formatting_prompts,
    args=training_args,
    peft_config=peft_config,

)

metrics = trainer.evaluate()       # 注意这里没有参数
metrics["epoch"] = 0
trainer.log(metrics)

trainer.train()

# 保存最终模型
save_dir = f"{output_dir}/lora_adapter"
trainer.model.save_pretrained(save_dir)   # 保存 LoRA 权重
tokenizer.save_pretrained(save_dir)       # tokenizer 可选一起存

print("LoRA adapter saved to:", save_dir)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/5704 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5704 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5704 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5704 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/1427 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Model Preparation Time,Entropy,Num Tokens,Mean Token Accuracy
10,1.427100,1.421979,0.009500,1.223101,309891.000000,0.697828
20,1.380800,1.334356,0.009500,1.204583,620612.000000,0.717177
30,1.285100,1.233756,0.009500,1.169619,939488.000000,0.728295
40,1.171100,1.116274,0.009500,1.112801,1256882.000000,0.752416
50,1.055800,0.991619,0.009500,1.003601,1572188.000000,0.775680
60,0.931600,0.881166,0.009500,0.877105,1888148.000000,0.796465
70,0.830100,0.813458,0.009500,0.814521,2199674.000000,0.810984
80,0.781800,0.781652,0.009500,0.771038,2514248.000000,0.814311
90,0.751000,0.758745,0.009500,0.772977,2828010.000000,0.817957
100,0.739300,0.741015,0.009500,0.756825,3146986.000000,0.821113


LoRA adapter saved to: /content/drive/MyDrive/Columbia/nlp/final/model/qwen25_math_1_5b_lora/lora_adapter


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
adapter_path = "/content/drive/.../qwen25_math_1_5b_sft/lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Loaded base + LoRA adapter")

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnBoxedAnswer(StoppingCriteria):
    """Halts generation once a \\boxed{} answer is produced."""

    def __init__(self, tokenizer, prompt_token_len: int):
        self.tokenizer = tokenizer
        self.prompt_token_len = prompt_token_len

    def _has_boxed_answer(self, text: str) -> bool:
        idx = text.rfind("\\boxed{")
        if idx == -1:
            return False
        return "}" in text[idx:]

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        generated_ids = input_ids[0][self.prompt_token_len :]
        if generated_ids.numel() == 0:
            return False
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        if self._has_boxed_answer(text):
            return True
        return False

trainer.model.eval()
device = trainer.model.device
print(device)

cuda:0


In [ ]:
SYSTEM_PROMPT = (
    "You are a mathematical reasoning assistant. "
    "Solve problems using step-by-step reasoning with clear explanations. "
    "Always provide your final answer in \\boxed{} format."
)

def build_user_prompt(q: str) -> str:
    return q + "\nPlease reason step by step, and put your final answer within \\boxed{}."

In [20]:
import re
from fractions import Fraction

def normalize_to_fraction(ans: str):
    # 转成字符串
    if ans is None:
        return None
    if not isinstance(ans, str):
        ans = str(ans)

    s = ans.replace(",", "").strip()
    if s == "":
        return None               # 👈 直接返回 None，表示不可解析

    # 去掉 "####" 之类前缀
    s = re.sub(r"^#+\s*", "", s).strip()
    if s == "":
        return None

    # 只取第一个 token，防止类似 "42 ." / "42 apples"
    parts = s.split()
    if len(parts) == 0:
        return None
    s = parts[0]

    # 分数形式 a/b
    if "/" in s:
        try:
            num, den = s.split("/")
            return Fraction(int(num), int(den))
        except Exception:
            pass

    # 整数 / 小数
    try:
        return Fraction(s)
    except Exception:
        try:
            return Fraction(str(float(s)))
        except Exception:
            return None

In [ ]:
import pandas as pd

df_test = pd.read_csv("/content/gsm8k_test.csv")  # 你自己的路径
print(df_test.head()[["question", "final_answer"]])

model = trainer.model
correct = 0
total = len(df_test)

pred_answers = []
gold_answers = []

for idx, row in df_test.iterrows():
    question = row["question"]
    gold = str(row["final_answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question)},
    ]

    # 用和训练时一样的 chat template，只到 user 为止
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 👈 让模型接着生成 assistant
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_length = inputs['input_ids'].shape[1]
    stopping_criteria = StoppingCriteriaList([StopOnBoxedAnswer(tokenizer, prompt_length)])

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            stopping_criteria=stopping_criteria,
        )

    # 只取新生成部分
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    pred = extract_pred_answer(generated_text)
    pred_answers.append(pred)
    gold_answers.append(gold)

    norm_pred = normalize_to_fraction(pred)
    norm_gold = normalize_to_fraction(gold)

    is_correct = (norm_pred is not None and norm_gold is not None and norm_pred == norm_gold)
    correct += int(is_correct)

    # 可以只打印前几条看看
    if idx < 5:
        print(f"\n#{idx}")
        print("Q:", question)
        print("Model output:\n", generated_text[:400], "...")
        print(f"gold={gold}, pred={pred} --> {'✓' if is_correct else '✗'}")

acc = correct / total * 100
print(f"\nEval on {total} examples: accuracy = {correct}/{total} = {acc:.2f}%")

                                            question final_answer
0  Janet’s ducks lay 16 eggs per day. She eats th...           18
1  A robe takes 2 bolts of blue fiber and half th...            3
2  Josh decides to try flipping a house.  He buys...        70000
3  James decides to run 3 sprints 3 times a week....          540
4  Every day, Wendi feeds each of her chickens th...           20

#0
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Model output:
 <think>
Let me understand the problem: Janet has ducks that lay 16 eggs per day. She eats 3 eggs for breakfast every morning, and bakes 4 eggs for friends. She sells the rest for $2 per egg. We need to find out how much she makes daily.

First, calculate the total eggs laid: 16 eggs/day.

Breakfast: 

IndexError: list index out of range

In [24]:
from tqdm.auto import tqdm

In [ ]:
df_test = pd.read_csv("/content/gsm8k_test.csv")
print(df_test.head()[["question", "final_answer"]])

model = trainer.model
model.eval()
device = model.device

batch_size = 32  # A100 可以再往上调，比如 32

correct = 0
total = len(df_test)

pred_answers = []
gold_answers = []

# 用 tqdm 包一下 range，这样就能看到进度了
for start in tqdm(range(0, total, batch_size), desc="Evaluating"):
    end = min(start + batch_size, total)
    batch = df_test.iloc[start:end]

    questions = batch["question"].tolist()
    golds = [str(x) for x in batch["final_answer"].tolist()]
    gold_answers.extend(golds)

    # 1) 构造 prompts
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(q)},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)

    # 2) 一次性 tokenize + generate
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    # Note: StoppingCriteria with batch generation can be tricky - only stop individual sequences
    # For batch processing, we keep max_new_tokens reasonable and let it generate fully
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 3) 拆成单条样本，抽答案
    input_ids = inputs["input_ids"]
    batch_preds = []

    for i in range(len(questions)):
        input_len = (input_ids[i] != tokenizer.pad_token_id).sum().item()
        gen_ids = outputs[i][input_len:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        pred = extract_pred_answer(gen_text)
        batch_preds.append(pred)

        global_idx = start + i
        if global_idx < 3:  # 只看前几条
            print(f"\n#{global_idx}")
            print("Q:", questions[i])
            print("Model output:\n", gen_text[:400], "...")
            print(f"gold={golds[i]}, pred={pred}")

    pred_answers.extend(batch_preds)

    # 4) 这一 batch 统计正确数，并在 tqdm 上显示当前 acc
    batch_correct = 0
    for pred, gold in zip(batch_preds, golds):
        norm_pred = normalize_to_fraction(pred)
        norm_gold = normalize_to_fraction(gold)
        is_correct = (
            norm_pred is not None
            and norm_gold is not None
            and norm_pred == norm_gold
        )
        correct += int(is_correct)
        batch_correct += int(is_correct)

    current_acc = correct / (start + len(batch)) * 100
    tqdm.write(f"Processed {start + len(batch)}/{total}, acc = {current_acc:.2f}%")

acc = correct / total * 100
print(f"\nFinal accuracy on {total} examples: {correct}/{total} = {acc:.2f}%")

                                            question final_answer
0  Janet’s ducks lay 16 eggs per day. She eats th...           18
1  A robe takes 2 bolts of blue fiber and half th...            3
2  Josh decides to try flipping a house.  He buys...        70000
3  James decides to run 3 sprints 3 times a week....          540
4  Every day, Wendi feeds each of her chickens th...           20


Evaluating:   0%|          | 0/83 [00:00<?, ?it/s]


#0
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Model output:
 2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Please think step by step and then give the final answer in \boxed{} on the last line.
assistant
<think>
Let me understand the problem: Janet has ducks that lay 16 eggs per day. She eats 3 eggs for breakfast every morning (7 days a week, so 3 eggs/day). She bakes 4 eggs for friends every day (same 7 days a w ...
gold=18, pred=

#1
Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Model output:
  reasoning and math notation. The final answer must be exact (no decimal approximation) and placed in \boxed{} on the last line.
user
Problem:
A robe

KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
import torch

df_test = pd.read_csv("/content/gsm8k_test.csv")
print(df_test.head()[["question", "final_answer"]])

model = trainer.model
model.eval()
device = model.device

# 推理阶段用左 padding，避免 warning
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

batch_size = 128  # A100 可以根据显存调大一点

total = len(df_test)
correct = 0
seen = 0

save_path = "/content/gsm8k_test_pred_qwen25_math_lora_stream.csv"

# 如果你想每次从头重新评估，先把旧文件删掉
if os.path.exists(save_path):
    os.remove(save_path)

for start in tqdm(range(0, total, batch_size), desc="Evaluating"):
    end = min(start + batch_size, total)
    batch = df_test.iloc[start:end]

    questions = batch["question"].tolist()
    golds = [str(x) for x in batch["final_answer"].tolist()]

    # 1) 构造 prompts
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(q)},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt)

    # 2) 批量 tokenize + generate
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    # Note: For batch generation, we don't use StoppingCriteria to avoid complexity
    # Instead, we rely on max_new_tokens to control generation length
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2056,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_ids = inputs["input_ids"]

    batch_preds = []
    batch_gens = []
    batch_correct_flags = []

    # 3) 拆开每个样本，抽答案 & 统计
    for i in range(len(questions)):
        input_len = (input_ids[i] != tokenizer.pad_token_id).sum().item()
        gen_ids = outputs[i][input_len:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        pred = extract_pred_answer(gen_text)
        gold = golds[i]

        norm_pred = normalize_to_fraction(pred)
        norm_gold = normalize_to_fraction(gold)
        is_corr = (
            norm_pred is not None
            and norm_gold is not None
            and norm_pred == norm_gold
        )

        batch_preds.append(pred)
        batch_gens.append(gen_text)
        batch_correct_flags.append(int(is_corr))

        correct += int(is_corr)
        seen += 1

        global_idx = start + i
        if global_idx < 3:
            print(f"\n#{global_idx}")
            print("Q:", questions[i])
            print("Model output:\n", gen_text[:400], "...")
            print(f"gold={gold}, pred={pred}, correct={is_corr}")

    current_acc = correct / seen * 100
    tqdm.write(f"Processed {seen}/{total}, acc = {current_acc:.2f}%")

    # 4) 这一批写入 CSV（追加写）
    batch_df = pd.DataFrame({
        "index": range(start, end),
        "question": questions,
        "gold": golds,
        "pred": batch_preds,
        "is_correct": batch_correct_flags,
        "generation": batch_gens,
    })

    # 第一个 batch 写 header，后面都 append
    write_header = not os.path.exists(save_path)
    batch_df.to_csv(
        save_path,
        mode="a",
        header=write_header,
        index=False,
    )

# 5) 最终整体准确率（即使中途崩了，前面的 batch 已经写入 save_path）
final_acc = correct / seen * 100
print(f"\nFinal accuracy on {seen} / {total} examples: {correct}/{seen} = {final_acc:.2f}%")
print("Stream results saved to:", save_path)

                                            question final_answer
0  Janet’s ducks lay 16 eggs per day. She eats th...           18
1  A robe takes 2 bolts of blue fiber and half th...            3
2  Josh decides to try flipping a house.  He buys...        70000
3  James decides to run 3 sprints 3 times a week....          540
4  Every day, Wendi feeds each of her chickens th...           20


Evaluating:   0%|          | 0/11 [00:00<?, ?it/s]